# Airbnb Data Analysis

Saya akan menganalisis data Airbnb dari Kaggle untuk memahami karakteristik data hotel nya

In [102]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 1. LOAD DATASET AIRBNB

Dataset AirBnb akan dibaca menggunakan Pandas.

In [ ]:
df = pd.read_csv('dataset/airbnb.csv')

df.head()

## 2. DATA OVERVIEW

Memahami terlebih dahulu tentang struktur dataset Airbnb

In [ ]:
df.shape
df.info()

## 3. DATA QUALITY CHECK

Check the quality of the dataset

The main checks include:
- Missing values
- Duplicate records
- Data types
- Invalid or inconsistent values

### 3.1 Missing Values

Check the number of missing values in each column.
Missing value may affect calculations and visualizations, so they need to be identified before further analysis.

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
missing_percentage = (
    df.isna().mean() * 100
).sort_values(ascending=False)

missing_percentage

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count" : missing_values,
    "missing_percentage" : missing_percentage
})

missing_summary = (
    missing_summary.sort_values("missing_percentage", ascending=False)
)

missing_summary

### 3.2 DUPLICATE RECORDS

Check whether the dataset contains duplicate records.
Duplicate records can affect aggregation and statistical analysis.

In [ ]:
df.duplicated().sum()

### 3.3 DATA TYPES

Review data type of each column to make sure numerical, categorical, and date-related columns are represented appropriately.

In [ ]:
df.info()

### 3.4 NUMERICAL VARIABLES

Identify numerical columns that can be used for statistical analysis and visualization.

In [ ]:
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
numeric_columns
df[numeric_columns].describe().T

### 3.5 CATEGORICAL VARIABLES

Categorical variables will be reviewed to understand the different groups or categories available in the dataset.

In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns
categorical_columns

for column in categorical_columns:
    print(f"{column}: {df[column].nunique()} unique values")

## 4. DATA CLEANING

Based on the data quality checks, several issues need to be addressed before performing exploratory analysis.

The cleaning process will focus on:
- Cleaning Price
- Handling missing values
- Removing duplicate records
- Correcting data types where necessary

### 4.1 Cleaning Price

The dollar sign will be removed from the `price` values, and the column
will be converted from text to a numerical data type.

In [ ]:
df["price"] = (
    df["price"].str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
df["price"].dtype

### 4.2 REMOVING DUPLICATE RECORDS

Duplicate records can cause certain listings to be counted more than
once. Since fully duplicated rows do not provide additional information,
they will be removed.

In [ ]:
df = df.drop_duplicates().copy()
df.duplicated().sum()

### 4.3 HANDLING MISSING VALUES

Missing values will be handled based on the meaning of each variables

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df = df.dropna(
    subset=[
        "price", "neighbourhood"
    ]
).copy()

df[["price", "neighbourhood"]].isna().sum().sort_values(ascending=False)

### 4.4 FINAL DATA QUALITY CHECKS

After cleaning missing values, a final data quality check is performed to make sure the dataset is ready for exploratory analysis.

The final checks include:
- Dataset dimensions
- Duplicate records
- Missing values in key analytical columns
- Data type of the price column
- Price statistics

In [99]:
print("Dataset shape:", df.shape)
print("Duplicate records:", df.duplicated().sum())
print("Missing price:", df["price"].isna().sum())
print("Missing neighbourhood:", df["neighbourhood"].isna().sum())
print("Price data type:", df["price"].dtype)

Dataset shape: (101796, 26)
Duplicate records: 0
Missing price: 0
Missing neighbourhood: 0
Price data type: float64


### 4.6 CREATE CLEAN DATASET

A seperate cleaned DataFrame is created to preserve the original dataset while providing a stable dataset for analysis and dashboard preparation.

In [100]:
df_clean = df.copy()
df_clean.shape
df_clean.head()

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


## 5. EXPLORATORY DATA ANALYSIS

Explore the Airbnb listings to identify patterns and relationships that may provide useful business insights.

The analysis will focus on:
- Pricing
- Neighbourhoods
- Reviews
- Availability
- Minimum nights
- Host activity

### Question 1 : What does the Airbnb price distribution look like?

Understanding the price distribution can help identify the typical price range and detect unusually expensive or inexpensive listings

In [ ]:
df_clean["price"].describe()

In [ ]:
plt.figure(figsize=(10, 6))

df_clean["price"].plot(kind='hist', bins=50, edgecolor='black', alpha=0.7)

plt.title("Distribution of Airbnb Prices")
plt.xlabel("Price")
plt.ylabel("Number of Listings")
plt.show()

### Insight

The Airbnb price distribution appears relatively uniform across the observed price range, with a similar number of listings across most price intervals.
The listings are distributed across a wide range of prices, from approximately $50 to $1200. There is no clear concentration of listings at a specific price range based on this histogram.

### Question 2 : Which neighbourhoods have the highest average Airbnb prices?

Location can be an important factor in accomodation pricing.
This analysis compares the average Airbnb price across neighbourhoods to identify areas with relatively higher or lower prices.

In [ ]:
avg_price_by_neighbourhood = df_clean.groupby("neighbourhood")["price"].mean().sort_values(ascending=False)

avg_price_by_neighbourhood.head(10)

In [ ]:
plt.figure(figsize=(10, 6))

avg_price_by_neighbourhood.head(10).sort_values().plot(kind='barh', color='skyblue', edgecolor='black')

plt.title("Top 10 Neighbourhoods by Average Airbnb Price")
plt.xlabel("Average Price")
plt.ylabel("Neighbourhood")
plt.show()

### Insight

The average Airbnb price varies across neighbourhoods.

The neighbourhood with the highest average price is **New Dorp**, while **Riverdale** has the lowest average price among the top 10 neighbourhoods shown.

### Checking Listing Volume

Average prices can be misleading when a neighbourhood has only a small number of listings. Therefore, I will also examine the number of listings in each neighbourhood.

In [ ]:
listings_count_by_neighbourhood = (
    df_clean.groupby("neighbourhood").size().sort_values(ascending=False)
)

listings_count_by_neighbourhood.head(10)

In [ ]:
neighbourhood_summary = (
    df_clean.groupby("neighbourhood").agg(
        average_price=("price", "mean"),
        listings_count=("neighbourhood", "size")
    ).sort_values("average_price", ascending=False)
)

neighbourhood_summary.head(10)

In [ ]:
neighbourhood_summary_50 = (
    neighbourhood_summary[
        neighbourhood_summary["listings_count"] >= 50
    ].sort_values("average_price", ascending=False)
)

neighbourhood_summary_50.head(10)

In [ ]:
plt.figure(figsize=(12, 6))

neighbourhood_summary_50.head(10).sort_values("average_price").plot(kind='barh')

plt.title("Top 10 Neighbourhoods by Average Airbnb Price (with at least 50 listings)")
plt.xlabel("Average Price")
plt.ylabel("Neighbourhood")
plt.show()

### Insight

Among neighbourhoods with at least 50 listings, average Airbnb prices vary considerably across locations.

The highest-priced neighbourhoods may represent areas with stronger demand, premium properties, or other location-related characteristics.

However, neighborhood alone may not explain the differences in price.
Further analysis should examine room type and other listing characteristics
to determine what factors may contribute to higher Airbnb prices.

### Question 3 : How does Airbnb price vary by room type?

Room type can influence the price of an Airbnb listing. This analysis compares the average price and number of listings across different room types.

In [ ]:
df_clean["room type"].value_counts()

### Comparing Average Price by Room Type

Now I will calculate the average price for each room type to compare their pricing levels.

In [ ]:
avg_price_by_room_type = df_clean.groupby("room type")["price"].mean().sort_values(ascending=False)
avg_price_by_room_type

In [ ]:
plt.figure(figsize=(12, 6))

avg_price_by_room_type.plot(kind='barh', edgecolor='black')

plt.title("Average Airbnb Price by Room Type")
plt.xlabel("Average Price")
plt.ylabel("Room Type")
plt.show()

### Comparing Listing Volume

To put the average prices into context, I will also compare the number of listings available for each room type.

In [ ]:
listing_count_by_room_type = (df_clean["room type"].value_counts())
listing_count_by_room_type

In [ ]:
room_type_summary = (
    df_clean.groupby("room type").agg(
        average_price=("price", "mean"),
        median_price=("price", "median"),
        listings_count=("room type", "size")
    ).sort_values("average_price", ascending=False)
)

room_type_summary

In [ ]:
plt.figure(figsize=(12, 6))

listing_count_by_room_type.sort_values().plot(kind='bar', color='lightgreen', edgecolor='black')

plt.title("Number of Airbnb Listings by Room Type")
plt.xlabel("Room Type")
plt.ylabel("Number of Listings")
plt.show()

### Insight

The average Airbnb prices varies across room types.

**Entire home/apt** has the highest average price, while **Hotel room* has the lowest average price.

The difference in pricing suggest that room type may be an important factor associated with Airbnb prices. However, the relationship should be interpreted together with other factors such as neighbourhood and availability.

## 6. DASHBOARD DATA PREPARATION

The analysis results are transformed into structured datasets that can be consumed by the React dashboard.

Instead of sending the entire dataset to the frontend, only the aggregated results required for visualization will be exported.


In [103]:
overview = {
    "total_listings": int(df_clean.shape[0]),
    "average_price": round(df_clean["price"].mean(), 2),
    "median_price": round(df_clean["price"].median(), 2),
    "neighbourhoods": int(df_clean["neighbourhood"].nunique())
}

overview

{'total_listings': 101796,
 'average_price': np.float64(625.34),
 'median_price': np.float64(624.0),
 'neighbourhoods': 224}

In [106]:
price_counts, price_bins = np.histogram(df_clean["price"], bins=50)
price_distribution = [
    {
        "price": int(price_bins[i]),
        "count": int(price_counts[i])
    }

    for i in range(len(price_counts))
]

price_distribution[:10]

[{'price': 50, 'count': 1990},
 {'price': 73, 'count': 2038},
 {'price': 96, 'count': 1999},
 {'price': 119, 'count': 2057},
 {'price': 142, 'count': 2080},
 {'price': 165, 'count': 1932},
 {'price': 188, 'count': 2107},
 {'price': 211, 'count': 1969},
 {'price': 234, 'count': 2050},
 {'price': 257, 'count': 2024}]

In [107]:
top_neighbourhoods = (
    neighbourhood_summary_50.head(10).reset_index()
)

top_neighbourhoods

,neighbourhood,average_price,listings_count
0,Columbia St,712.227273,88
1,Gravesend,705.611511,140
2,Briarwood,703.579832,121
3,Stapleton,699.234375,65
4,Coney Island,695.260000,50
5,Kew Gardens Hills,690.393939,66
6,Cambria Heights,686.487179,78
7,Throgs Neck,684.696429,56
8,Stuyvesant Town,680.915663,83
9,Highbridge,676.029851,67


In [108]:
neighbourhood_data = (
    top_neighbourhoods[
        [
            "neighbourhood",
            "average_price",
            "listings_count"
        ]
    ]
    .round(2)
    .to_dict(orient="records")
)

neighbourhood_data

[{'neighbourhood': 'Columbia St',
  'average_price': 712.23,
  'listings_count': 88},
 {'neighbourhood': 'Gravesend',
  'average_price': 705.61,
  'listings_count': 140},
 {'neighbourhood': 'Briarwood',
  'average_price': 703.58,
  'listings_count': 121},
 {'neighbourhood': 'Stapleton', 'average_price': 699.23, 'listings_count': 65},
 {'neighbourhood': 'Coney Island',
  'average_price': 695.26,
  'listings_count': 50},
 {'neighbourhood': 'Kew Gardens Hills',
  'average_price': 690.39,
  'listings_count': 66},
 {'neighbourhood': 'Cambria Heights',
  'average_price': 686.49,
  'listings_count': 78},
 {'neighbourhood': 'Throgs Neck',
  'average_price': 684.7,
  'listings_count': 56},
 {'neighbourhood': 'Stuyvesant Town',
  'average_price': 680.92,
  'listings_count': 83},
 {'neighbourhood': 'Highbridge',
  'average_price': 676.03,
  'listings_count': 67}]

In [109]:
room_type_data = (
    room_type_summary.reset_index().round(2).to_dict(orient="records")
)

room_type_data

[{'room type': 'Hotel room',
  'average_price': 666.39,
  'median_price': 643.0,
  'listings_count': 115},
 {'room type': 'Shared room',
  'average_price': 634.83,
  'median_price': 651.5,
  'listings_count': 2208},
 {'room type': 'Entire home/apt',
  'average_price': 625.24,
  'median_price': 624.0,
  'listings_count': 53429},
 {'room type': 'Private room',
  'average_price': 624.94,
  'median_price': 623.0,
  'listings_count': 46306}]

In [110]:
dashboard_data = {
    "overview": overview,
    "price_distribution": price_distribution,
    "room_type": room_type_data,
    "neighbourhood": neighbourhood_data
}

dashboard_data.keys()

dict_keys(['overview', 'price_distribution', 'room_type', 'neighbourhood'])

## 7. EXPORT DASHBOARD DATA

The aggregated analysis results are exported as a JSON file for use in the react dashboard.

Only summarized data is exported instead of the complete dataset.

In [111]:
import json

In [112]:
with open(
    "dataset/airbnb_dashboard.json", "w", encoding="utf-8"
) as f:
    json.dump(dashboard_data, f, ensure_ascii=False, indent=2)

In [113]:
with open(
    "dataset/airbnb_dashboard.json",
    "r",
    encoding="utf-8"
) as f:
    dashboard_check = json.load(f)